# MODIS Feature Extraction (MOD13Q1)

Extracts MODIS features described in the dataset overview. By default, overlapping reflectance bands with Landsat (red/NIR/blue) are excluded.

In [10]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
import xarray as xr

from datetime import date
from tqdm import tqdm
import os
import time
import random
import certifi

os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
os.environ["CURL_CA_BUNDLE"] = certifi.where()

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Exclude overlap with Landsat reflectance bands by default
INCLUDE_OVERLAP = False

tqdm.pandas()

In [11]:
# Optional: install dependencies if missing
!pip install numpy pandas odc-stac pystac-client planetary-computer tqdm certifi


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [12]:
CATALOG = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

BASE_FEATURES = [
    "ndvi",
    "evi",
    "mir",
    "vi_quality",
    "pixel_reliability",
    "sun_zenith",
    "view_zenith",
    "rel_azimuth",
    "day_of_year",
]

OVERLAP_FEATURES = ["red", "nir", "blue"]

FEATURES = BASE_FEATURES + (OVERLAP_FEATURES if INCLUDE_OVERLAP else [])

ASSET_CANDIDATES = {
    "ndvi": ["ndvi"],
    "evi": ["evi"],
    "red": ["red", "sur_refl_b01", "b01"],
    "nir": ["nir", "sur_refl_b02", "b02"],
    "blue": ["blue", "sur_refl_b03", "b03"],
    "mir": ["mir", "swir", "sur_refl_b07", "b07"],
    "vi_quality": ["vi_quality", "vi quality", "quality"],
    "pixel_reliability": ["pixel_reliability", "pixel reliability", "reliability"],
    "sun_zenith": ["sun_zenith", "solar_zenith", "sun zenith"],
    "view_zenith": ["view_zenith", "view zenith"],
    "rel_azimuth": ["relative_azimuth", "rel_azimuth", "azimuth"],
    "day_of_year": ["day_of_year", "day of year", "doy"],
}


def _match_asset_key(item, candidates):
    asset_keys = list(item.assets.keys())
    for cand in candidates:
        if cand in asset_keys:
            return cand
        cand_lower = cand.lower()
        for key in asset_keys:
            if cand_lower in key.lower():
                return key
    return None


def _build_asset_map(item):
    asset_map = {}
    for feature in FEATURES:
        asset_map[feature] = _match_asset_key(item, ASSET_CANDIDATES[feature])
    return asset_map


def _chunk_bbox(df: pd.DataFrame, buffer_deg: float = 0.02):
    min_lat = df['Latitude'].min() - buffer_deg
    max_lat = df['Latitude'].max() + buffer_deg
    min_lon = df['Longitude'].min() - buffer_deg
    max_lon = df['Longitude'].max() + buffer_deg
    return [min_lon, min_lat, max_lon, max_lat]


def _chunk_datetime_range(df: pd.DataFrame):
    times = pd.to_datetime(df['Sample Date'], dayfirst=True, errors='coerce')
    if times.notna().any():
        min_date = times.min() - pd.Timedelta(days=32)
        max_date = times.max() + pd.Timedelta(days=32)
    else:
        min_date = pd.Timestamp('2011-01-01')
        max_date = pd.Timestamp('2015-12-31')
    return f"{min_date.date()}/{max_date.date()}"


def _select_point_values(data: xr.Dataset, df: pd.DataFrame, asset_map: dict) -> pd.DataFrame:
    times = pd.to_datetime(df['Sample Date'], dayfirst=True, errors='coerce')
    times_filled = times.fillna(pd.Timestamp('2011-01-01'))
    lats = df['Latitude'].values
    lons = df['Longitude'].values

    x_name = 'x' if 'x' in data.coords else 'lon'
    y_name = 'y' if 'y' in data.coords else 'lat'

    points = xr.Dataset({
        x_name: (('points',), lons),
        y_name: (('points',), lats),
        'time': (('points',), times_filled.values),
    })

    sel = data.sel(
        **{x_name: points[x_name], y_name: points[y_name], 'time': points['time']},
        method='nearest',
    )

    mask = times.isna() | pd.isna(lats) | pd.isna(lons)

    out = {}
    for feature in FEATURES:
        asset_key = asset_map.get(feature)
        if asset_key and asset_key in sel:
            vals = sel[asset_key].values.astype(float)
        else:
            vals = np.full(len(df), np.nan, dtype=float)
        if mask.any():
            vals[mask.values] = np.nan
        out[feature] = vals

    # Fallback day_of_year from selected time if asset missing
    if 'day_of_year' in FEATURES and (asset_map.get('day_of_year') not in sel):
        try:
            out['day_of_year'] = pd.to_datetime(sel['time'].values).dayofyear
        except Exception:
            pass

    return pd.DataFrame(out)


def load_chunk_data(chunk_df: pd.DataFrame, max_retries: int = 3):
    bbox = _chunk_bbox(chunk_df)
    datetime_range = _chunk_datetime_range(chunk_df)

    for attempt in range(1, max_retries + 1):
        try:
            search = CATALOG.search(
                collections=["modis-13Q1-061"],
                bbox=bbox,
                datetime=datetime_range,
            )
            items = list(search.item_collection())
            if not items:
                return None, {}

            signed_items = [pc.sign(item) for item in items]
            asset_map = _build_asset_map(signed_items[0])
            available_bands = [key for key in asset_map.values() if key]
            if not available_bands:
                return None, asset_map

            data = stac_load(
                signed_items,
                bands=available_bands,
                bbox=bbox,
                crs="EPSG:4326",
            )
            return data, asset_map

        except Exception as exc:
            sleep_s = 2 ** attempt
            time.sleep(sleep_s)

    return None, {}


In [13]:
# Load data
Water_Quality_df = pd.read_csv(os.path.join(PROJECT_ROOT + "/Provided Datasets", 'water_quality_training_dataset.csv'))
Validation_df = pd.read_csv(os.path.join(PROJECT_ROOT, 'submission_template.csv'))

print(f"Training rows: {len(Water_Quality_df)}")
print(f"Validation rows: {len(Validation_df)}")

Training rows: 9319
Validation rows: 200


In [ ]:
chunk_size = 200
max_retries = 3

train_features_path = os.path.join(PROJECT_ROOT, 'New Datasets', 'modis_features_training_allvars.csv')
val_features_path = os.path.join(PROJECT_ROOT, 'New Datasets', 'modis_features_validation_allvars.csv')

expected_cols = ['Latitude', 'Longitude', 'Sample Date'] + FEATURES


def count_rows_in_csv(path: str) -> int:
    with open(path, 'r', encoding='utf-8') as f:
        return max(sum(1 for _ in f) - 1, 0)


def extract_chunked(input_df: pd.DataFrame, output_path: str, label: str):
    start_idx = 0
    if os.path.exists(output_path):
        existing_header = pd.read_csv(output_path, nrows=0).columns.tolist()
        if existing_header != expected_cols:
            raise ValueError(
                f"Existing file has different columns.\n"
                f"Expected: {expected_cols}\n"
                f"Found:    {existing_header}\n"
                f"Fix: delete/rename the existing file or update expected_cols."
            )
        start_idx = count_rows_in_csv(output_path)

    print(f"🚀 Running MODIS extraction for {label} (chunked)...")
    print(f"Total rows: {len(input_df)}")
    print(f"Output file: {output_path}")
    print(f"Chunk size: {chunk_size}")
    print(f"Resuming from row index: {start_idx}")

    for chunk_start in range(start_idx, len(input_df), chunk_size):
        chunk_end = min(chunk_start + chunk_size, len(input_df))
        chunk_df = input_df.iloc[chunk_start:chunk_end].copy()

        print(f"\nProcessing rows {chunk_start}..{chunk_end-1} ({len(chunk_df)} rows)")
        try:
            data, asset_map = load_chunk_data(chunk_df, max_retries=max_retries)
            if data is None:
                chunk_feats = pd.DataFrame({feature: np.nan for feature in FEATURES}, index=chunk_df.index)
            else:
                chunk_feats = _select_point_values(data, chunk_df, asset_map)

            chunk_feats['Latitude'] = chunk_df['Latitude'].values
            chunk_feats['Longitude'] = chunk_df['Longitude'].values
            chunk_feats['Sample Date'] = chunk_df['Sample Date'].values

            chunk_out = chunk_feats[expected_cols]
            write_header = (not os.path.exists(output_path)) or (count_rows_in_csv(output_path) == 0)
            chunk_out.to_csv(output_path, mode='a', header=write_header, index=False)

            time.sleep(0.5)

        except Exception as e:
            print(f"\n❌ Chunk failed at rows {chunk_start}..{chunk_end-1}: {e}")
            print("You can rerun this cell to resume from the last completed chunk.")
            break


extract_chunked(Water_Quality_df, train_features_path, "training")
extract_chunked(Validation_df, val_features_path, "validation")

if os.path.exists(train_features_path):
    display(pd.read_csv(train_features_path).head())
if os.path.exists(val_features_path):
    display(pd.read_csv(val_features_path).head())

🚀 Running MODIS extraction for training (chunked)...
Total rows: 9319
Output file: /Users/aaravsonthalia/Projects/Water-Quality-Prediction/New Datasets/modis_features_training_allvars.csv
Chunk size: 200
Resuming from row index: 0

Processing rows 0..199 (200 rows)
